In [1]:
#GOOGLE VERTEX
#not sure the difference between the two libs below, prompting seems to work with just the first but the gemini documentation says to do the second
#pip install google-cloud-aiplatform to install library
#pip install -q -U google-generativeai
# run this in google CLI to set up auth: gcloud auth application-default login
#pd.set_option('display.max_colwidth', None) to expand df cells

import vertexai
from vertexai.generative_models import GenerativeModel, ChatSession

In [3]:
#get the data to send
import pandas as pd
import json
# pd.set_option('display.max_colwidth', None) # if you want to view the full json blob in the printed dataframe, use this

file_path = r'C:\Users\ChristinePayton\OneDrive - USDA\Documents\Data\NLP_Public_Comments\letters_2020_2021.csv'
supplemental_file = r'C:\Users\ChristinePayton\OneDrive - USDA\Documents\Data\NLP_Public_Comments\additional_letter_info.csv'
output_path = r'C:\Users\ChristinePayton\OneDrive - USDA\Documents\Data\NLP_Public_Comments\letters_ai_extraction_modelcomp.csv'

# get csv
df = pd.read_csv(file_path, encoding="cp1252") #these files are encoded strangely, have to pass encoding value
supplemental = pd.read_csv(supplemental_file, encoding="cp1252")

# left join on the 'Letter Id' field
df = pd.merge(df, supplemental, on='Letter Id', how='left')

letter_types = ['Unique'] #the letter types we want from the supplemental data file, can comma separate multiple

#Filter for only 'unique' values (this dataset is 90% generated form duplicate data, don't want to use tokens on that) and letters that are not essentially null (these cause errors)
df = df[(df['Letter Type'].isin(letter_types)) & (df['Letter Text'].str.len() > 10)]

df = df.drop_duplicates(subset=['Author Name', 'Letter Text'])
df = df.sample(n=100) #get a random small number of rows for testing, remove this line later to run on full csv

df


,ProjectId,Project Name,Comment Period Id,Comment Period Type,Comment Period Name,Comment Period Start Year,Author Name,Author ZipCode,Letter Id,Letter Sequence Number,Letter Text,Letter Submission Date,Form Set,Letter Type,Delivery Type
151064,357,Nantahala and Pisgah NFs Plan Revision,3103,Notice of Availability,NaN,2020,Kent Wilcox,28718,2524210,4312,I am concerned about the current plan standard...,2020-06-23 13:33:46.0000000,NaN,Unique,CARA Web-portal
194590,1600,Stibnite Gold Project EIS,3562,Notice of Availability,NaN,2020,Jack Hurty,83702,2604147,2355,This comment is being submitted in opposition ...,2020-10-13 15:17:50.0000000,NaN,Unique,CARA Web-portal
124627,2619,FSM 7700 and 7710 E-bikes,3567,Other,E-bikes,2020,Darren Singer,97212,2644934,4308,I appreciate the challenges the USFS faces in ...,2020-10-23 05:46:10.0000000,NaN,Unique,CARA Web-portal
126526,2619,FSM 7700 and 7710 E-bikes,3567,Other,E-bikes,2020,Eric Schroeder,98294,2670611,7495,E-bikes should definitely be allowed wherever ...,2020-10-26 17:46:23.0000000,NaN,Unique,CARA Web-portal
154098,2572,Operation and Maintenance of Developed Recreat...,3504,Other,Directive,2020,Mike Hyde,84021,2536060,3,Please see attached letter.,2020-07-27 15:51:44.0000000,NaN,Unique,CARA Web-portal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121891,2619,FSM 7700 and 7710 E-bikes,3567,Other,E-bikes,2020,Ronald Everson,81321,2556509,201,I am a 70 year old mountain bike rider and I n...,2020-09-27 03:14:17.0000000,NaN,Unique,CARA Web-portal
122621,2619,FSM 7700 and 7710 E-bikes,3567,Other,E-bikes,2020,James Black,43082,2589562,975,As an avid rider on NSFS trails in West Virgin...,2020-10-08 21:46:46.0000000,NaN,Unique,CARA Web-portal
124531,2619,FSM 7700 and 7710 E-bikes,3567,Other,E-bikes,2020,Landon Arkens,53713,2644710,4141,Thank you for classifying all three classes of...,2020-10-23 00:39:37.0000000,NaN,Unique,CARA Web-portal
125530,2619,FSM 7700 and 7710 E-bikes,3567,Other,E-bikes,2020,Kristi Haphey,97045,2667374,6081,I oppose the use of e bikes on non motorized t...,2020-10-25 17:31:58.0000000,NaN,Unique,CARA Web-portal


In [2]:

project_id = "usfs-gcp-rand-test"
location = "us-central1"
vertexai.init(project=project_id, location=location)
model = GenerativeModel("gemini-1.0-pro")

In [4]:
def completion_iteration(value):
    try:

        dynamic_message_text = [
 "You are a feedback analyst. You parse and extract from submitted letters to help organize and structure the data." + user_prompt + json.dumps(example_json) + "Here is the letter text to analyze: " + value
        ]
        
        response = model.generate_content(dynamic_message_text)
        return response.text
    except Exception as e:
        #commenting this out for now as it's super verbose and massively scrolls the thing down
        #print(f"Error processing input '{value}': {str(e)}")
        return None


#categories as text, not array, because we are passing in message string
response_categories = "Air Quality, Botany, Climate Change, Cultural/Heritage, Facilities, FireFuels, Fisheries, Hydrology, Lands/Special Uses, Minerals/Geology, NEPA/Proj Development, Other/Misc, Public Engagement, Range/Weeds, Recreation, Roadless, Silviculture/Veg, SocioEconomic, Soils, Transportation, Visuals, Wilderness, Wildlife"

user_prompt = "Extract the following information: Sentiment (required, -1 to 1, where -1 is extremely negative and 1 is extremely positive), SentimentConfidence (decimal, 0-1, represents the confidence level of your sentiment number), Category (required, single value, choices are: " + response_categories + ". CitingLaw (boolean, return True if the text implies that a law is being broken or is referencing a law), BriefSummary (required, 1-2 sentence summary of the letter), SpecificFeedback (boolean, required, true if a specific resolution to their issue is identified), ProposedResolution (summarization of what the submitter thinks will resolve their issue). Your response should be ONLY the JSON analysis, no other text. Here is an example response: "

#Giving the AI an example is required here to get a consistent output
example_json = {
  "Sentiment": -0.5,
  "SentimentConfidence": .8,
  "Category": "Hydrology",
  "CitingLaw": True,
  "BriefSummary": "The letter expresses concerns over river pollution impacting community health and calls for stricter regulatory oversight and sustainable practices.",
  "SpecificFeedback": True,
  "ProposedResolution":"Stricter regulation on industry waste dumping."
  #you could also have it propose a response that people could review/use as a starting point here
  #"SuggestedResponse":"Thank you for your thoughtful feedback regarding the dumping regulations. We share your concern about the need for robust measures to protect our environment and public health. Please know that your comments are invaluable to us and contribute significantly to our ongoing review process aimed at strengthening our regulations. We are actively working with various stakeholders, including environmental experts and community leaders, to ensure our policies effectively address these concerns while promoting sustainable practices. To stay engaged and informed about the progress and developments in this area, we encourage you to visit our website regularly and participate in upcoming public forums. Your active involvement is essential as we strive to enhance our environmental policies for the betterment of our community and future generations."
}
# Apply the function to all the rows in the dataframe
df['ResultGemini1'] = df["Letter Text"].apply(lambda value: completion_iteration(value=value))
df
#this took 2 min 22 sec to run on 100 rows

C:\Users\ChristinePayton\AppData\Roaming\Python\Python310\site-packages\google\auth\_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


,ProjectId,Project Name,Comment Period Id,Comment Period Type,Comment Period Name,Comment Period Start Year,Author Name,Author ZipCode,Letter Id,Letter Sequence Number,Letter Text,Letter Submission Date,Form Set,Letter Type,Delivery Type,ResultGemini1
151064,357,Nantahala and Pisgah NFs Plan Revision,3103,Notice of Availability,NaN,2020,Kent Wilcox,28718,2524210,4312,I am concerned about the current plan standard...,2020-06-23 13:33:46.0000000,NaN,Unique,CARA Web-portal,"{""Sentiment"": -0.5, ""SentimentConfidence"": 0.8..."
194590,1600,Stibnite Gold Project EIS,3562,Notice of Availability,NaN,2020,Jack Hurty,83702,2604147,2355,This comment is being submitted in opposition ...,2020-10-13 15:17:50.0000000,NaN,Unique,CARA Web-portal,"{""Sentiment"": -1, ""SentimentConfidence"": 0.9, ..."
124627,2619,FSM 7700 and 7710 E-bikes,3567,Other,E-bikes,2020,Darren Singer,97212,2644934,4308,I appreciate the challenges the USFS faces in ...,2020-10-23 05:46:10.0000000,NaN,Unique,CARA Web-portal,"{""Sentiment"": 0, ""SentimentConfidence"": 0.7, ""..."
126526,2619,FSM 7700 and 7710 E-bikes,3567,Other,E-bikes,2020,Eric Schroeder,98294,2670611,7495,E-bikes should definitely be allowed wherever ...,2020-10-26 17:46:23.0000000,NaN,Unique,CARA Web-portal,"{""Sentiment"": 1, ""SentimentConfidence"": 0.9, ""..."
154098,2572,Operation and Maintenance of Developed Recreat...,3504,Other,Directive,2020,Mike Hyde,84021,2536060,3,Please see attached letter.,2020-07-27 15:51:44.0000000,NaN,Unique,CARA Web-portal,"{""Sentiment"": 0.0, ""SentimentConfidence"": 0.0,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121891,2619,FSM 7700 and 7710 E-bikes,3567,Other,E-bikes,2020,Ronald Everson,81321,2556509,201,I am a 70 year old mountain bike rider and I n...,2020-09-27 03:14:17.0000000,NaN,Unique,CARA Web-portal,"{""Sentiment"": 1, ""SentimentConfidence"": 0.7, ""..."
122621,2619,FSM 7700 and 7710 E-bikes,3567,Other,E-bikes,2020,James Black,43082,2589562,975,As an avid rider on NSFS trails in West Virgin...,2020-10-08 21:46:46.0000000,NaN,Unique,CARA Web-portal,"{""Sentiment"": -0.8, ""SentimentConfidence"": 0.9..."
124531,2619,FSM 7700 and 7710 E-bikes,3567,Other,E-bikes,2020,Landon Arkens,53713,2644710,4141,Thank you for classifying all three classes of...,2020-10-23 00:39:37.0000000,NaN,Unique,CARA Web-portal,"{""Sentiment"": 1, ""SentimentConfidence"": 1, ""Ca..."
125530,2619,FSM 7700 and 7710 E-bikes,3567,Other,E-bikes,2020,Kristi Haphey,97045,2667374,6081,I oppose the use of e bikes on non motorized t...,2020-10-25 17:31:58.0000000,NaN,Unique,CARA Web-portal,"{""Sentiment"": -1, ""SentimentConfidence"": 0.8, ..."


In [6]:
#GPT 3.5 ITERATE OVER DATAFRAME - CONNECT WITH VPN BEFORE RUNNING!
import os
from openai import AzureOpenAI
import json


client = AzureOpenAI(
    azure_endpoint = "https://oai-nonprd-openai-poc-01.openai.azure.com", 
    api_key = os.getenv("AZURE_OPENAI_KEY"),
    api_version="2024-02-15-preview"
)

def completion_iteration_az(value):
    try:
        # This is the structure for GPT 3.5, other models may have different syntax
        dynamic_message_text = [
            {"role": "system", "content": "You are a feedback analyst. You parse and extract from submitted letters to help organize and structure the data."},
            {"role": "user", "content": user_prompt + json.dumps(example_json) + "Here is the letter text to analyze: " + value}
        ]
        
        completion = client.chat.completions.create(
            model="BASE-gpt-35-turbo",
            messages=dynamic_message_text,  # Use the dynamically constructed message
            temperature=0.6,
            max_tokens=800,
            top_p=0.95,
            frequency_penalty=0,
            presence_penalty=0,
            stop=None
        )
        return completion.choices[0].message.content
    except Exception as e:
        #commenting this out for now as it's super verbose and massively scrolls the thing down
        #print(f"Error processing input '{value}': {str(e)}")
        return None
#These were the suggested categories for response triage
#categories as text, not array, because we are passing in message string
response_categories = "Air Quality, Botany, Climate Change, Cultural/Heritage, Facilities, FireFuels, Fisheries, Hydrology, Lands/Special Uses, Minerals/Geology, NEPA/Proj Development, Other/Misc, Public Engagement, Range/Weeds, Recreation, Roadless, Silviculture/Veg, SocioEconomic, Soils, Transportation, Visuals, Wilderness, Wildlife"

user_prompt = "Extract the following information: Sentiment (required, -1 to 1, where -1 is extremely negative and 1 is extremely positive), SentimentConfidence (decimal, 0-1, represents the confidence level of your sentiment number), Category (required, single value, choices are: " + response_categories + ". CitingLaw (boolean, return True if the text implies that a law is being broken or is referencing a law), BriefSummary (required, 1-2 sentence summary of the letter), SpecificFeedback (boolean, required, true if a specific resolution to their issue is identified), ProposedResolution (summarization of what the submitter thinks will resolve their issue). Your response should be ONLY the JSON analysis, no other text. Here is an example response: "

#Giving the AI an example is required here to get a consistent output
example_json = {
  "Sentiment": -0.5,
  "SentimentConfidence": .8,
  "Category": "Hydrology",
  "CitingLaw": True,
  #"Tags": ["water quality", "pollution", "regulation"],
  "BriefSummary": "The letter expresses concerns over river pollution impacting community health and calls for stricter regulatory oversight and sustainable practices.",
  "SpecificFeedback": True,
  "ProposedResolution":"Stricter regulation on industry waste dumping."
  #you could also have it propose a response that people could review/use as a starting point here
  #"SuggestedResponse":"Thank you for your thoughtful feedback regarding the dumping regulations. We share your concern about the need for robust measures to protect our environment and public health. Please know that your comments are invaluable to us and contribute significantly to our ongoing review process aimed at strengthening our regulations. We are actively working with various stakeholders, including environmental experts and community leaders, to ensure our policies effectively address these concerns while promoting sustainable practices. To stay engaged and informed about the progress and developments in this area, we encourage you to visit our website regularly and participate in upcoming public forums. Your active involvement is essential as we strive to enhance our environmental policies for the betterment of our community and future generations."
}
# Apply the function to all the rows in the dataframe
df['ResultGPT35'] = df["Letter Text"].apply(lambda value: completion_iteration_az(value=value))
df

#runs 2 min 9s on 100 rows

,ProjectId,Project Name,Comment Period Id,Comment Period Type,Comment Period Name,Comment Period Start Year,Author Name,Author ZipCode,Letter Id,Letter Sequence Number,Letter Text,Letter Submission Date,Form Set,Letter Type,Delivery Type,ResultGemini1,ResultGPT35
151064,357,Nantahala and Pisgah NFs Plan Revision,3103,Notice of Availability,NaN,2020,Kent Wilcox,28718,2524210,4312,I am concerned about the current plan standard...,2020-06-23 13:33:46.0000000,NaN,Unique,CARA Web-portal,"{""Sentiment"": -0.5, ""SentimentConfidence"": 0.8...","{""Sentiment"": -0.5, ""SentimentConfidence"": 0.8..."
194590,1600,Stibnite Gold Project EIS,3562,Notice of Availability,NaN,2020,Jack Hurty,83702,2604147,2355,This comment is being submitted in opposition ...,2020-10-13 15:17:50.0000000,NaN,Unique,CARA Web-portal,"{""Sentiment"": -1, ""SentimentConfidence"": 0.9, ...","{""Sentiment"": -0.8, ""SentimentConfidence"": 0.9..."
124627,2619,FSM 7700 and 7710 E-bikes,3567,Other,E-bikes,2020,Darren Singer,97212,2644934,4308,I appreciate the challenges the USFS faces in ...,2020-10-23 05:46:10.0000000,NaN,Unique,CARA Web-portal,"{""Sentiment"": 0, ""SentimentConfidence"": 0.7, ""...","{""Sentiment"": 0.2, ""SentimentConfidence"": 0.7,..."
126526,2619,FSM 7700 and 7710 E-bikes,3567,Other,E-bikes,2020,Eric Schroeder,98294,2670611,7495,E-bikes should definitely be allowed wherever ...,2020-10-26 17:46:23.0000000,NaN,Unique,CARA Web-portal,"{""Sentiment"": 1, ""SentimentConfidence"": 0.9, ""...","{""Sentiment"": 0.5, ""SentimentConfidence"": 0.9,..."
154098,2572,Operation and Maintenance of Developed Recreat...,3504,Other,Directive,2020,Mike Hyde,84021,2536060,3,Please see attached letter.,2020-07-27 15:51:44.0000000,NaN,Unique,CARA Web-portal,"{""Sentiment"": 0.0, ""SentimentConfidence"": 0.0,...","{""Sentiment"": null, ""SentimentConfidence"": nul..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121891,2619,FSM 7700 and 7710 E-bikes,3567,Other,E-bikes,2020,Ronald Everson,81321,2556509,201,I am a 70 year old mountain bike rider and I n...,2020-09-27 03:14:17.0000000,NaN,Unique,CARA Web-portal,"{""Sentiment"": 1, ""SentimentConfidence"": 0.7, ""...","{""Sentiment"": 0.9, ""SentimentConfidence"": 0.9,..."
122621,2619,FSM 7700 and 7710 E-bikes,3567,Other,E-bikes,2020,James Black,43082,2589562,975,As an avid rider on NSFS trails in West Virgin...,2020-10-08 21:46:46.0000000,NaN,Unique,CARA Web-portal,"{""Sentiment"": -0.8, ""SentimentConfidence"": 0.9...","{""Sentiment"": -0.7, ""SentimentConfidence"": 0.9..."
124531,2619,FSM 7700 and 7710 E-bikes,3567,Other,E-bikes,2020,Landon Arkens,53713,2644710,4141,Thank you for classifying all three classes of...,2020-10-23 00:39:37.0000000,NaN,Unique,CARA Web-portal,"{""Sentiment"": 1, ""SentimentConfidence"": 1, ""Ca...","{""Sentiment"": 0.1, ""SentimentConfidence"": 0.6,..."
125530,2619,FSM 7700 and 7710 E-bikes,3567,Other,E-bikes,2020,Kristi Haphey,97045,2667374,6081,I oppose the use of e bikes on non motorized t...,2020-10-25 17:31:58.0000000,NaN,Unique,CARA Web-portal,"{""Sentiment"": -1, ""SentimentConfidence"": 0.8, ...","{""Sentiment"": -0.8, ""SentimentConfidence"": 0.9..."


In [7]:
#output to csv
df_output = df[['Letter Id', 'ResultGPT35', 'ResultGemini1']]
df_output.to_csv(output_path, index=False)
df

,ProjectId,Project Name,Comment Period Id,Comment Period Type,Comment Period Name,Comment Period Start Year,Author Name,Author ZipCode,Letter Id,Letter Sequence Number,Letter Text,Letter Submission Date,Form Set,Letter Type,Delivery Type,ResultGemini1,ResultGPT35
151064,357,Nantahala and Pisgah NFs Plan Revision,3103,Notice of Availability,NaN,2020,Kent Wilcox,28718,2524210,4312,I am concerned about the current plan standard...,2020-06-23 13:33:46.0000000,NaN,Unique,CARA Web-portal,"{""Sentiment"": -0.5, ""SentimentConfidence"": 0.8...","{""Sentiment"": -0.5, ""SentimentConfidence"": 0.8..."
194590,1600,Stibnite Gold Project EIS,3562,Notice of Availability,NaN,2020,Jack Hurty,83702,2604147,2355,This comment is being submitted in opposition ...,2020-10-13 15:17:50.0000000,NaN,Unique,CARA Web-portal,"{""Sentiment"": -1, ""SentimentConfidence"": 0.9, ...","{""Sentiment"": -0.8, ""SentimentConfidence"": 0.9..."
124627,2619,FSM 7700 and 7710 E-bikes,3567,Other,E-bikes,2020,Darren Singer,97212,2644934,4308,I appreciate the challenges the USFS faces in ...,2020-10-23 05:46:10.0000000,NaN,Unique,CARA Web-portal,"{""Sentiment"": 0, ""SentimentConfidence"": 0.7, ""...","{""Sentiment"": 0.2, ""SentimentConfidence"": 0.7,..."
126526,2619,FSM 7700 and 7710 E-bikes,3567,Other,E-bikes,2020,Eric Schroeder,98294,2670611,7495,E-bikes should definitely be allowed wherever ...,2020-10-26 17:46:23.0000000,NaN,Unique,CARA Web-portal,"{""Sentiment"": 1, ""SentimentConfidence"": 0.9, ""...","{""Sentiment"": 0.5, ""SentimentConfidence"": 0.9,..."
154098,2572,Operation and Maintenance of Developed Recreat...,3504,Other,Directive,2020,Mike Hyde,84021,2536060,3,Please see attached letter.,2020-07-27 15:51:44.0000000,NaN,Unique,CARA Web-portal,"{""Sentiment"": 0.0, ""SentimentConfidence"": 0.0,...","{""Sentiment"": null, ""SentimentConfidence"": nul..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121891,2619,FSM 7700 and 7710 E-bikes,3567,Other,E-bikes,2020,Ronald Everson,81321,2556509,201,I am a 70 year old mountain bike rider and I n...,2020-09-27 03:14:17.0000000,NaN,Unique,CARA Web-portal,"{""Sentiment"": 1, ""SentimentConfidence"": 0.7, ""...","{""Sentiment"": 0.9, ""SentimentConfidence"": 0.9,..."
122621,2619,FSM 7700 and 7710 E-bikes,3567,Other,E-bikes,2020,James Black,43082,2589562,975,As an avid rider on NSFS trails in West Virgin...,2020-10-08 21:46:46.0000000,NaN,Unique,CARA Web-portal,"{""Sentiment"": -0.8, ""SentimentConfidence"": 0.9...","{""Sentiment"": -0.7, ""SentimentConfidence"": 0.9..."
124531,2619,FSM 7700 and 7710 E-bikes,3567,Other,E-bikes,2020,Landon Arkens,53713,2644710,4141,Thank you for classifying all three classes of...,2020-10-23 00:39:37.0000000,NaN,Unique,CARA Web-portal,"{""Sentiment"": 1, ""SentimentConfidence"": 1, ""Ca...","{""Sentiment"": 0.1, ""SentimentConfidence"": 0.6,..."
125530,2619,FSM 7700 and 7710 E-bikes,3567,Other,E-bikes,2020,Kristi Haphey,97045,2667374,6081,I oppose the use of e bikes on non motorized t...,2020-10-25 17:31:58.0000000,NaN,Unique,CARA Web-portal,"{""Sentiment"": -1, ""SentimentConfidence"": 0.8, ...","{""Sentiment"": -0.8, ""SentimentConfidence"": 0.9..."


In [10]:
#basic prompt
response = model.generate_content("What is the circumference of the earth?")

#reference output with response.text
print(response.text)


candidates {
  content {
    role: "model"
    parts {
      text: "40,075.017 kilometers"
    }
  }
  finish_reason: STOP
  safety_ratings {
    category: HARM_CATEGORY_HATE_SPEECH
    probability: NEGLIGIBLE
    probability_score: 0.0766838044
    severity: HARM_SEVERITY_NEGLIGIBLE
    severity_score: 0.102484219
  }
  safety_ratings {
    category: HARM_CATEGORY_DANGEROUS_CONTENT
    probability: NEGLIGIBLE
    probability_score: 0.128197357
    severity: HARM_SEVERITY_NEGLIGIBLE
    severity_score: 0.0792103186
  }
  safety_ratings {
    category: HARM_CATEGORY_HARASSMENT
    probability: NEGLIGIBLE
    probability_score: 0.0863234773
    severity: HARM_SEVERITY_NEGLIGIBLE
    severity_score: 0.046638038
  }
  safety_ratings {
    category: HARM_CATEGORY_SEXUALLY_EXPLICIT
    probability: NEGLIGIBLE
    probability_score: 0.0407692641
    severity: HARM_SEVERITY_NEGLIGIBLE
    severity_score: 0.0822539553
  }
}
usage_metadata {
  prompt_token_count: 8
  candidates_token_count: 11
 

AttributeError: 'GenerationResponse' object has no attribute 'safety_ratings'